In [0]:
%python
import numpy as np
import pandas as pd
from pyspark.sql.functions import sum as _sum

# 1. AGGREGATE BASELINE DATA
# We get the total volume and value of transactions marked as 'CLEAN'
baseline_metrics = spark.table("`prism-sentinel-stream`.prism_silver.transactions_refined") \
    .filter("risk_flag = 'CLEAN'") \
    .select(_sum("amount").alias("total_value")).collect()[0]

total_clean_value = float(baseline_metrics["total_value"])

# 2. MONTE CARLO SIMULATION LOGIC
def run_risk_simulation(total_value, iterations=1000):
    # Assumption: Mean evasion rate is 0.5% (0.005) with a standard deviation of 0.1%
    evasion_rates = np.random.normal(0.005, 0.001, iterations)
    simulated_losses = evasion_rates * total_value
    
    return simulated_losses

# 3. EXECUTE SIMULATION
print(f"🎲 Running Monte Carlo on ${total_clean_value:,.2f} of 'Clean' transactions...")
results = run_risk_simulation(total_clean_value)

# 4. CAPTURE STATISTICAL OUTCOMES
sim_df = pd.DataFrame(results, columns=['potential_exposure'])
summary = {
    "Average Potential Exposure": sim_df['potential_exposure'].mean(),
    "95th Percentile (Worst Case)": sim_df['potential_exposure'].quantile(0.95),
    "Minimum Exposure": sim_df['potential_exposure'].min()
}

# 5. PERSIST TO GOLD TABLE
gold_table = "`prism-sentinel-stream`.prism_gold.risk_simulation_results"
spark.createDataFrame(sim_df).write.format("delta").mode("overwrite").saveAsTable(gold_table)

print("\n--- Simulation Complete ---")
for key, value in summary.items():
    print(f"{key}: ${value:,.2f}")